# 19. Remove Nth Node From End of List
**Difficulty:** 🟡 Medium · **Topic:** Linked List · **LeetCode:** https://leetcode.com/problems/remove-nth-node-from-end-of-list/

## 💡 Concepts

**Core concept(s):** A **gap of n** between two pointers turns 'from the end' into 'at the current spot'.

**Why it applies here:** You can't index from the end of a singly linked list. But if a fast pointer starts n nodes ahead of a slow pointer, when fast reaches the end the slow pointer sits just before the node to remove — one pass, no length count.

**Key intuition:** Send one pointer n steps ahead; when it hits the end, the other is right before the target.

---

### 📚 What is a Linked List?
A **linked list** is a chain of nodes; each node holds a value and a pointer to the **next** node. Unlike an array there is no index — you can only walk forward from the head.
- **In Python:** a small `ListNode` class with `.val` and `.next`.

### 📚 Pointers & the Dummy Node
Linked-list code moves **pointers** (references to nodes). A **dummy** node placed before the head removes annoying "is this the first node?" special cases — you build off `dummy.next` and return it at the end.

### 📚 Fast & Slow Pointers
Two pointers moving at different speeds: **slow** one step, **fast** two. They meet inside a loop (cycle detection) and the slow one lands on the middle when fast reaches the end.
- **Complexity:** one pass, **O(1)** extra space.

---

**Prerequisite knowledge:**
- Dummy node.
- Two pointers with a fixed gap.

## 📝 Problem

Remove the n-th node from the **end** and return the head.

**Example**
```
1->2->3->4->5, n=2  ->  1->2->3->5
```

> Two approaches, both `O(n)`: two-pass (count then delete) and one-pass (gap pointers).

In [ ]:
from typing import Optional, List

class ListNode:
    """A node in a singly linked list: a value plus a link to the next node."""
    def __init__(self, val=0, next=None):
        self.val = val                     # the value stored at this node
        self.next = next                   # link to the next node (None at the end)

def build_list(vals):
    """Turn a Python list into a linked list; return its head."""
    dummy = ListNode(); cur = dummy        # dummy node avoids special-casing the first node
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    """Turn a linked list back into a Python list (handy for printing / assertions)."""
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

### Approach 1 — Two Pass (worst)

**Idea:** Count the length, then walk to the node before the target and unlink it.

**Time:** `O(n)` (two passes). **Space:** `O(1)`.

In [ ]:
def remove_nth_two_pass(head, n):
    length = 0; cur = head
    while cur:                             # pass 1: count how many nodes there are
        length += 1; cur = cur.next
    if n == length:
        return head.next                   # removing the very first node
    cur = head
    for _ in range(length - n - 1):        # pass 2: walk to the node BEFORE the target
        cur = cur.next
    cur.next = cur.next.next               # skip over (delete) the target node
    return head

### Approach 2 — One Pass with Gap (optimal)

**Idea:** Put `fast` n nodes ahead of `slow` (both start at a dummy). Advance together; when `fast` hits the end, `slow` is just before the target.

**Time:** `O(n)` single pass. **Space:** `O(1)`.

In [ ]:
def remove_nth_one_pass(head, n):
    dummy = ListNode(0, head)              # dummy handles the "remove the head" case cleanly
    fast = slow = dummy
    for _ in range(n):
        fast = fast.next                   # move fast n nodes ahead -> a gap of n
    while fast.next:                       # move both until fast reaches the last node
        fast = fast.next; slow = slow.next
    slow.next = slow.next.next             # slow is right before the target -> unlink it
    return dummy.next

In [ ]:
# Correctness check
tests = [([1,2,3,4,5],2,[1,2,3,5]), ([1],1,[]), ([1,2],1,[1]), ([1,2],2,[2])]
for vals, n, exp in tests:
    assert to_list(remove_nth_two_pass(build_list(vals), n)) == exp
    assert to_list(remove_nth_one_pass(build_list(vals), n)) == exp
    print(vals, "n=", n, "->", exp)
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio when `n` → `2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_list(list(range(n))), 1)
solutions = {
    "two-pass O(n)": remove_nth_two_pass,
    "one-pass O(n)": remove_nth_one_pass,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Fixed-gap two pointers:** convert "n from the end" into "here" in a single pass.
- **Dummy node:** removes the "delete the head" special case.
- **Signal:** "n-th from the end", "delete/find relative to the tail".
- **Related problems:** Middle of the Linked List, Rotate List, Reorder List.
- **Common pitfalls:** (1) off-by-one on the gap; (2) not using a dummy when the head may be removed.